# Share results between apps with a workspace bucket

Every Workbench app runs on its own virtual machine, so a file saved on one app's disk is invisible to
other apps and to other users, and is lost when the app is deleted. **Workspace buckets** are the
shared, persistent layer: anything written there is visible to every app and every member of the
workspace, and is governed by the workspace's policies.

This notebook writes a small result from JupyterLab to the `shared-work` bucket, then shows how to read
it back from **another app**: a second notebook, the R Analysis app, VS Code, or the command line.

**Time:** about 2 minutes.

## 1. Find the bucket
Workspace resources are looked up by name, never by hardcoded bucket path.

In [ ]:
import io
import subprocess
from datetime import datetime, timezone

import pandas as pd
from google.cloud import storage


def resolve(resource_name):
    return subprocess.run(["wb", "resolve", "--name", resource_name], capture_output=True, text=True, check=True).stdout.strip()


BUCKET_URL = resolve("shared-work")                       # gs://<bucket>[/<folder>]
bucket_name, _, base_prefix = BUCKET_URL.removeprefix("gs://").partition("/")
bucket = storage.Client().bucket(bucket_name)
OBJECT = f"{base_prefix + '/' if base_prefix else ''}examples/share-demo/hello.csv"
print(f"gs://{bucket_name}/{OBJECT}")

## 2. Produce a result and write it to the bucket
The Cloud Storage client uploads the file directly from memory; nothing is kept on the app's disk.

In [ ]:
result = pd.DataFrame({
    "written_by": ["JupyterLab"],
    "written_at_utc": [datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")],
    "message": ["Hello from the Python app. Any app in this workspace can read this file."],
})
bucket.blob(OBJECT).upload_from_string(result.to_csv(index=False), content_type="text/csv")
print(f"Wrote gs://{bucket_name}/{OBJECT}")

## 3. Read it back here, then from somewhere else

In [ ]:
pd.read_csv(io.BytesIO(bucket.blob(OBJECT).download_as_bytes()))

Now open the **same file from a different app** in this workspace:

**R Analysis app**
```r
bucket <- system2("wb", c("resolve", "--name", "shared-work"), stdout = TRUE)
df <- read.csv(pipe(paste("gcloud storage cat", file.path(bucket, "examples/share-demo/hello.csv"))))
df
```

**VS Code or any terminal**
```bash
gcloud storage cat "$(wb resolve --name shared-work)/examples/share-demo/hello.csv"
```

**Workbench UI**: Resources → `shared-work` → `examples/share-demo/hello.csv`.

## What to remember
- App disk is private and temporary; buckets are shared and permanent.
- Buckets, BigQuery datasets, and Git repos are the three ways apps and people share work in a workspace.
- Access to the bucket follows workspace roles: Readers can read it, Writers and Owners can write.

## Provenance

Generate information about this notebook environment and the packages installed.

In [ ]:
!date

Conda and pip installed packages:

In [ ]:
!conda env export

JupyterLab extensions:

In [ ]:
!jupyter labextension list

Number of cores:

In [ ]:
!grep ^processor /proc/cpuinfo | wc -l

Memory:

In [ ]:
!grep "^MemTotal:" /proc/meminfo

---
Copyright 2026 Verily Life Sciences LLC

Use of this source code is governed by a BSD-style
license that can be found in the LICENSE file or at
https://developers.google.com/open-source/licenses/bsd